# Reference Construction for DLPFC

In [ ]:
import query_construction_utils_01 as qcu
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
import mygene

%matplotlib inline
import matplotlib.pyplot as plt

## 1. Select 'Aging Cohort'

<div class="alert alert-block alert-info">
<b> I use "backed = 'r'" because the orignal object has 6 million+ cells and would not open properly in this notebook otherwise

</div>

In [ ]:
adata = ad.read_h5ad("/tscc/lustre/ddn/scratch/aopatel/dlpfc_reference/merged_final_clean.h5ad", backed='r')

# Check what columns exist
print(adata.obs.columns.tolist())

In [ ]:
print(adata.obs['Aging'].value_counts())

<div class="alert alert-block alert-info">
<b> I bring only the 'Aging cohort' since 1 million+ cells can be opened in this notebook.
</div>

In [ ]:
lifespan = adata[adata.obs['Aging'] == True].to_memory()

In [ ]:
# Ensure not a view 
print(lifespan.is_view)

In [ ]:
print(lifespan.obs['SubID_export_synapse'].nunique())  # how many unique donors?
print(lifespan.obs['Source'].value_counts())           # which brain banks?

## 2. Load metadata 

In [ ]:
meta = pd.read_csv("/tscc/lustre/ddn/scratch/aopatel/dlpfc_reference/NPS-AD_individual_metadata.csv")


In [ ]:
meta.columns.tolist()   # see all column names
meta.shape              # (n_donors, n_columns)
meta.dtypes             # check if age is stored as int/float/string

In [ ]:
# Ensure the ID columns for lifespan (adata) and meta (metadata file) match
print(lifespan.obs['SubID_export_synapse'].unique()[:5])
print(meta['individualID'].unique()[:5])

In [ ]:
meta_indexed = meta.set_index('individualID')

lifespan.obs = lifespan.obs.join(meta_indexed, on='SubID_export_synapse')

# Verify it worked
lifespan.obs[['SubID_export_synapse', 'ageDeath', 'PMI']].head()

## 3. Filter reference to keep donors of interest 

In [ ]:
# Fix ageDeath to numeric
lifespan.obs['ageDeath_numeric'] = lifespan.obs['ageDeath'].replace('90+', '91').astype(float)

# Nuclei per donor
nuclei_per_donor = lifespan.obs.groupby('SubID_export_synapse').size()
keep_nuclei = nuclei_per_donor[nuclei_per_donor >= 700].index.tolist()

# Apply all filters
filtered = lifespan[
    (lifespan.obs['Tardive_dyskinesia'] == 'No') &
    (lifespan.obs['PMI'] <= 20) &
    (lifespan.obs['ageDeath_numeric'] >= 40) &
    (lifespan.obs['SubID_export_synapse'].isin(keep_nuclei))
].copy()

print("Is filtered a view: ", filtered.is_view)  # True = view, False = copy
print("Cells:", filtered.shape[0])
print("Donors:", filtered.obs['SubID_export_synapse'].nunique())
print(filtered.obs['Source'].value_counts())

## 4. Quality Control (QC) and Preprocessing

In [ ]:
# Get rid of the normalized layer, and use the raw layer 
filtered.X = filtered.layers['counts'].copy()

In [ ]:
adata_ref=qcu.quality_controller(filtered,min_genes=1000,is_indexed=False)

In [ ]:
#### Violin plot visualization
sc.pl.violin(
            adata_ref,
            ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo", "pct_counts_hb",'pct_counts_in_top_20_genes'],
            jitter=0.4,
            multi_panel=True,
            show=True
        )

    

<div class="alert alert-block alert-info">
<b> PsychAD removed all mt, ribo, and hb genes upstream, likely for proccessing reasons

</div>

In [ ]:
adata_ref

In [ ]:
#### Filter genes that are not expressed in at least x cells 
sc.pp.filter_genes(adata_ref, min_cells=25)

print(f"Number of genes after filtering: {adata_ref.shape[1]}")
print(f"Number of cells after filtering: {adata_ref.shape[0]}")

## 5. Convert Ensemble IDs into Gene Symbols

### 5A. Query mygene

In [ ]:
mg = mygene.MyGeneInfo()

ensembl_ids = adata_ref.var_names.tolist()

# Query all at once - mygene handles batching internally
result = mg.querymany(
    ensembl_ids,
    scopes='ensembl.gene',
    fields='symbol',
    species='human',
    as_dataframe=True
)

print(result.head())
print(result.columns.tolist())
print(f"Total mappings: {len(result)}")
print(f"Missing symbols: {result['symbol'].isna().sum()}")

In [ ]:
# Reset index to get query column
result_reset = result.reset_index()

# Find duplicate query terms (Ensembl IDs that returned 2 hits)
dup_queries = result_reset[result_reset.duplicated(subset=['query'], keep=False)][['query', 'symbol']].sort_values('query')
print(f"Number of duplicate Ensembl IDs: {dup_queries['query'].nunique()}")
print(dup_queries.to_string())

### 5B. Filter ref for genes that do not have gene symbols and make index col. gene symbols

In [ ]:
# Get genes that have a valid symbol
result_clean = (
    result.reset_index()
    .dropna(subset=['symbol'])
    .drop_duplicates(subset=['query'], keep=False)  # ← drop ALL duplicate query rows
    .drop_duplicates(subset=['symbol'], keep='first')
)
print(f"Genes with unique symbol: {len(result_clean)}")
print(f"Genes without symbol (will be dropped): {len(adata_ref.var_names) - len(result_clean)}")

# Subset reference to only genes with symbols
genes_with_symbol = result_clean['query'].tolist()
adata_ref = adata_ref[:, genes_with_symbol].copy()

# Now rename
id_to_symbol = result_clean.set_index('query')['symbol'].to_dict()
adata_ref.var_names = [id_to_symbol[g] for g in adata_ref.var_names]
print(f"Reference after filtering: {adata_ref.n_vars} genes")
print(f"Example var_names: {adata_ref.var_names[:5].tolist()}")

## 6. Finishing touches

In [ ]:
adata_ref.obs = adata_ref.obs.rename(columns={'subclass': 'subclass_label'})


In [ ]:
#### Remove samples/cells without any cell type annotation
print("Empty count: ", adata_ref.obs['subclass_label'].isna().sum())

adata_ref = adata_ref[~adata_ref.obs['subclass_label'].isna()].copy()

print("Empty count (after removal): ", adata_ref.obs['subclass_label'].isna().sum())

In [ ]:
#### Save final reference file
adata_ref.write_h5ad("/tscc/lustre/ddn/scratch/aopatel/dlpfc_reference/adata_ref_DLPFC.h5ad")